# Enterprise Brain — ingestion and GraphRAG

Markdown in → **Qdrant** (meaning) + **Neo4j** (structure) → an agent that can use both.

| Piece | Where |
|---|---|
| Graph | Neo4j at `bolt://localhost:7687`, browser on <http://localhost:7474> |
| Vectors | Qdrant at <http://localhost:6333/dashboard> |
| Chat model | OpenAI (`CHAT_MODEL`, default `gpt-4.1-mini`) |
| Embeddings | OpenAI (`text-embedding-3-small`, 1536 dims) |
| Agent | Microsoft Agent Framework `ChatAgent` |
| UI | Gradio, launched from section 9 |

Before running: `cp .env.example .env`, fill in the keys, then `docker compose up -d neo4j qdrant`.

## 1 · Setup and health check

Nothing below works if this cell shows anything other than `ok`.

In [ ]:
import sys, asyncio
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from brain import config, clients

print(config.describe())
print()
for component, state in clients.health().items():
    print(f"{component:16s} {state}")

## 2 · The raw corpus

Seven markdown files that look like the real thing: a CRM account export, a field visit note,
a contract, an SAP equipment record, an opportunity, a second customer, and a competitor profile.

The interesting answers are the ones that need **more than one of these files at once** —
that is the whole argument for the graph.

In [ ]:
from brain import ingest

docs = ingest.load_documents()
for d in docs:
    print(f"{d.path:34s} {d.meta.get('source_system','?'):20s} {len(d.body):5d} chars  {d.title}")

In [ ]:
# What one document looks like after frontmatter parsing
print(docs[1].title, "|", docs[1].meta)
print(docs[1].body[:600], "...")

## 3 · Chunking

Split on `##` headings first — in business documents the heading is real structure, not decoration —
then hard split anything still over `CHUNK_CHARS`. The document title is prepended to every chunk so
the customer name travels with the text into the embedding.

In [ ]:
chunks = ingest.chunk_document(docs[1])
print(f"{len(chunks)} chunks from {docs[1].path}\n")
for c in chunks:
    print(f"  {c.chunk_id:28s} [{c.heading}] {len(c.text)} chars")

print("\n--- first chunk ---\n")
print(chunks[0].text[:500])

## 4 · Open schema extraction — one chunk first

You chose an **open schema**: the model decides the entity and relationship types instead of being
handed an ontology. Run it on a single chunk and look at what it invents before you let it loose
on the whole corpus.

If the types look chaotic across runs, that is the known cost of open schema. Two cheap fixes:
lower the temperature (already 0), or paste the output of `retrieval.graph_schema()` into the
extraction prompt as "types already in use" so it converges.

In [ ]:
import json

sample = ingest.extract_from_chunk(chunks[0], docs[1])
print(json.dumps(sample, indent=2, ensure_ascii=False))

## 5 · Full ingestion

For each chunk: embed → Qdrant, extract → Neo4j.

The join between the two stores is the chunk id: it is the Qdrant payload key **and** the `:Chunk`
node id, and every extracted edge carries the `chunk_id` it came from. That is what makes
provenance and hybrid retrieval possible later.

`reset=True` wipes both stores. Takes a couple of minutes — one LLM call per chunk.

In [ ]:
stats = ingest.ingest_all(reset=True)
stats

## 6 · What the graph actually looks like

Open <http://localhost:7474> and run `MATCH (n) RETURN n LIMIT 200` for the visual version.
Here is the text version.

In [ ]:
from brain import retrieval

print(retrieval.graph_schema())

In [ ]:
# Node counts by label
retrieval.run_cypher('''
MATCH (e:Entity)
UNWIND labels(e) AS label
WITH label WHERE label <> 'Entity'
RETURN label, count(*) AS entities
ORDER BY entities DESC
''')

## 7 · Cypher: the questions vector search cannot answer

Each of these needs *structure*. A pure RAG system would have to get lucky with a passage that
happens to state the whole chain in one place.

**7.1 — Everything we know about one customer, in one hop**

In [ ]:
retrieval.run_cypher('''
MATCH (c:Entity)-[r]-(other:Entity)
WHERE toLower(c.name) CONTAINS 'nordvind'
RETURN c.name AS entity, type(r) AS relationship, other.name AS connected, other.type AS kind
ORDER BY relationship
''')

**7.2 — Multi-hop: customer → site → equipment → issue**

This is the hop chain from the pitch. No single document contains all four.

In [ ]:
retrieval.run_cypher('''
MATCH path = (c:Entity)-[*1..4]-(i:Entity)
WHERE toLower(c.name) CONTAINS 'nordvind'
  AND (toLower(i.name) CONTAINS 'vibration' OR toLower(i.type) CONTAINS 'issue')
RETURN [n IN nodes(path) | n.name] AS hops, length(path) AS hop_count
ORDER BY hop_count
LIMIT 5
''')

**7.3 — Provenance: which file does each fact come from?**

Every edge stores the chunk it was extracted from, so the graph can always point back at a document.
This is the answer to "show me where this came from", and it is the reason management trusts the output.

In [ ]:
retrieval.run_cypher('''
MATCH (a:Entity)-[r]->(b:Entity)
WHERE r.chunk_id IS NOT NULL
MATCH (c:Chunk {id: r.chunk_id})<-[:HAS_CHUNK]-(d:Document)
RETURN a.name AS source, type(r) AS relationship, b.name AS target,
       d.path AS from_file, r.evidence AS evidence
LIMIT 12
''')

**7.4 — Cross-document: entities mentioned in more than one source system**

The single strongest signal that the brain is doing something no folder can: the same entity
arriving from CRM, SAP, a contract and a field note, and being recognised as one thing.

In [ ]:
retrieval.run_cypher('''
MATCH (d:Document)-[:HAS_CHUNK]->(:Chunk)-[:MENTIONS]->(e:Entity)
WITH e, collect(DISTINCT d.source_system) AS systems, collect(DISTINCT d.path) AS files
WHERE size(systems) > 1
RETURN e.name AS entity, e.type AS kind, systems, files
ORDER BY size(systems) DESC
LIMIT 10
''')

**7.5 — The neighbourhood helper**

Same idea wrapped as a function, because the agent will call it as a tool.

In [ ]:
for row in retrieval.neighbourhood("P-40", hops=2, limit=15):
    print(f"{row['source']} -[{row['relationship']}]-> {row['target']}")

## 8 · Three retrieval modes, side by side

Same question, three routes. Read the outputs and the difference sells itself.

In [ ]:
QUESTION = "Does our service agreement cover what is actually wrong with pump line P-40?"

# (a) vectors only — good at wording, blind to structure
for hit in retrieval.vector_search(QUESTION, k=3):
    print(f"[{hit['score']}] {hit['path']} — {hit['heading']}")
    print(hit['text'][:220].replace("\n", " "), "...\n")

In [ ]:
# (b) graph only — structure, no prose
retrieval.run_cypher('''
MATCH (e:Entity)-[r]-(other:Entity)
WHERE toLower(e.name) CONTAINS 'p-40' OR toLower(e.name) CONTAINS 'sla'
RETURN e.name AS entity, type(r) AS relationship, other.name AS connected, r.evidence AS evidence
LIMIT 15
''')

In [ ]:
# (c) hybrid: vector hit -> MENTIONS -> graph expansion, with sources on both
result = retrieval.hybrid_context(QUESTION, k=4, hops=2)
print(f"{len(result['chunks'])} passages, {len(result['facts'])} graph facts\n")
print(result["context"][:2500])

## 9 · Cypher *inside* RAG — a worked pattern

The pattern worth stealing: **Cypher narrows, vectors enrich, the model writes.**

1. Cypher finds the entities that satisfy a structural condition (here: an issue that touches
   equipment covered by a contract).
2. Those entity names become the vector query, so the passages retrieved are the ones about
   exactly those entities.
3. Both go to the model, with file paths, so the answer is checkable.

This is more reliable than letting a model write Cypher blind, and more precise than
retrieving on the user's raw wording.

In [ ]:
from brain.clients import complete

# step 1 — structural narrowing
seed_rows = retrieval.run_cypher('''
MATCH (equipment:Entity)
WHERE toLower(equipment.name) CONTAINS 'vibrat' OR toLower(equipment.description) CONTAINS 'vibrat'
OPTIONAL MATCH (equipment)-[r1]-(issue:Entity)
WHERE toLower(issue.name) CONTAINS 'vibrat' OR toLower(issue.description) CONTAINS 'vibrat'
OPTIONAL MATCH (equipment)-[r2]-(contract:Entity)
WHERE toLower(contract.name) CONTAINS 'sla' OR toLower(contract.type) CONTAINS 'contract'
RETURN DISTINCT equipment.name AS equipment, issue.name AS issue, contract.name AS contract
LIMIT 10
''')
seed_rows

In [ ]:
# step 2 — the graph result becomes the retrieval query
seed_terms = " ".join(
    {str(v) for row in seed_rows for v in row.values() if v}
)
passages = retrieval.vector_search(seed_terms, k=5)

# step 3 — hand the model structure + prose, demand citations
context = (
    "STRUCTURED FACTS FROM THE GRAPH:\n"
    + "\n".join(f"- {row}" for row in seed_rows)
    + "\n\nPASSAGES:\n"
    + "\n\n".join(f"[{p['path']}] {p['text']}" for p in passages)
)

answer = complete(
    f"{context}\n\nQuestion: {QUESTION}\n\n"
    "Answer in under 120 words. Cite the file path in square brackets after each claim. "
    "If the contract excludes the work, say so plainly and name the clause.",
    system="You are the enterprise brain. Use only the given context.",
)
print(answer)

## 10 · The agent

Microsoft Agent Framework `ChatAgent` over OpenAI, with four tools:
`ask_the_brain` (hybrid), `search_documents` (vectors), `query_graph` (Cypher it writes itself),
`describe_graph` (schema). The instructions tell it when to reach for which.

In [ ]:
from brain.agent import build_agent

agent = build_agent()

result = await agent.run("Give me a two line summary of the Nordvind Energi account.")
print(result.text)

### 10.1 · A descriptive question — it should reach for vectors

In [ ]:
print((await agent.run(
    "What exactly did Mette Lindgren say about the vibration alarms during the August visit?"
)).text)

### 10.2 · A structural question — it should write Cypher

Watch it call `describe_graph` and then `query_graph`. This is the question a folder full of
documents cannot answer at all.

In [ ]:
print((await agent.run(
    "Which entities appear in documents from more than one source system? "
    "Use Cypher and show the systems."
)).text)

### 10.3 · The multi-hop question from the pitch

In [ ]:
print((await agent.run(
    "What is the risk on the Nordvind Energi account before the September specification deadline, "
    "and what should we do about it? Cite sources."
)).text)

### 10.4 · The cross-customer question

The answer lives in two different accounts' files. Nothing in either document mentions the other.

In [ ]:
print((await agent.run(
    "Have we seen this vibration symptom at any other customer, and what was the root cause there?"
)).text)

## 11 · Gradio UI

Launches on <http://127.0.0.1:7860>. `inline=True` embeds it in the notebook instead.

For the container version instead: `docker compose --profile app up -d rag`.

In [ ]:
from brain import ui

demo = ui.build_ui(agent)
demo.launch(inline=False)   # returns immediately; call demo.close() to stop

In [ ]:
# demo.close()

## 12 · Where this goes next

- **Ontology convergence** — feed `graph_schema()` back into the extraction prompt so the open
  schema stabilises, or freeze the good types into a fixed list once you see them.
- **Entity resolution** — right now two spellings of a name are two nodes. Add an alias pass, or
  embed entity names and merge on cosine similarity above a threshold.
- **Community summaries** — the other half of GraphRAG: cluster the graph, summarise each cluster,
  and answer global questions ("what are the themes across all accounts") from those summaries.
- **More sources** — the ingestion contract is just `Document(doc_id, path, title, meta, body)`.
  Word, PowerPoint, Excel and transcripts all reduce to that.
- **Access control** — the pitch promised Entra ID at query time. Tag `:Document` nodes with
  security groups and filter both the Cypher and the Qdrant payload on the caller's groups.